# 00 Ingest GUI Workflow (`adamacs_ingest_v2`)

Use the GUI-first workflow to configure and run ingestion jobs safely.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


## 1) Setup and DB connection

Set `DJ_HOST` and `DJ_USER` in your shell if needed. Password is prompted by the DataJoint connection flow.


In [ ]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')

repo_root = Path.cwd()
config_path = repo_root / 'dj_local_conf.json'

import datajoint as dj
if config_path.exists():
    dj.config.load(str(config_path))
else:
    print(f'Warning: config not found at {config_path}; relying on environment variables.')

print(f'Working directory: {repo_root}')
print(f'DataJoint version: {dj.__version__}')
print(f"DB prefix: {dj.config.get('custom', {}).get('database.prefix', '<unset>')}")
dj.conn()


## 2) Load ingest v2 helper

In [ ]:
import adamacs.helpers.adamacs_ingest_v2 as ai
print('Loaded:', ai.__name__)


## 3) Discover session folders

In [ ]:
import os

# ADAMACS notebook overrides (edit here for local runs).
# ADAMACS_DATE_FILTER examples:
#   *                      -> no date filtering
#   2025-01-20             -> exact date
#   >=2025-01-01           -> on/after date
#   <2025-02-01            -> before date
#   2025-01-01:2025-01-31  -> inclusive date range
#   2025-01                -> legacy substring fallback
os.environ.setdefault("ADAMACS_SESSION_FILTER", "*")
os.environ.setdefault("ADAMACS_DATE_FILTER", "*")
os.environ.setdefault("ADAMACS_LAUNCH_GUI", "0")

print("ADAMACS_SESSION_FILTER =", os.environ["ADAMACS_SESSION_FILTER"])
print("ADAMACS_DATE_FILTER    =", os.environ["ADAMACS_DATE_FILTER"])
print("ADAMACS_LAUNCH_GUI     =", os.environ["ADAMACS_LAUNCH_GUI"])


In [ ]:
import fnmatch
import os
import re
from datetime import date
from natsort import natsorted
from IPython.display import display, HTML


def _parse_date_token(token):
    token = token.strip()
    for fmt in ("%Y-%m-%d", "%Y_%m_%d", "%Y%m%d"):
        try:
            from datetime import datetime

            return datetime.strptime(token, fmt).date()
        except ValueError:
            pass
    return None


def _extract_session_date(session_name):
    patterns = [
        r"(?<!\d)(20\d{2})[-_](\d{2})[-_](\d{2})(?!\d)",
        r"(?<!\d)(20\d{2})(\d{2})(\d{2})(?!\d)",
    ]
    for pattern in patterns:
        match = re.search(pattern, session_name)
        if not match:
            continue
        year, month, day = map(int, match.groups())
        try:
            return date(year, month, day)
        except ValueError:
            continue
    return None


def _parse_date_filter(expr):
    expr = (expr or "*").strip()
    if expr in {"", "*"}:
        return {"mode": "all"}

    if ":" in expr and not expr.startswith((">", "<")):
        lo_raw, hi_raw = expr.split(":", 1)
        lo = _parse_date_token(lo_raw) if lo_raw.strip() else None
        hi = _parse_date_token(hi_raw) if hi_raw.strip() else None
        if lo is None and hi is None:
            raise ValueError(
                f"Invalid ADAMACS_DATE_FILTER range: {expr!r}. "
                "Expected YYYY-MM-DD:YYYY-MM-DD (either side can be omitted)."
            )
        return {"mode": "range", "lo": lo, "hi": hi}

    for op in (">=", "<=", ">", "<"):
        if expr.startswith(op):
            rhs = _parse_date_token(expr[len(op) :].strip())
            if rhs is None:
                raise ValueError(
                    f"Invalid ADAMACS_DATE_FILTER comparator: {expr!r}. "
                    "Expected forms like >=2025-01-01 or <2025-02-01."
                )
            return {"mode": "cmp", "op": op, "rhs": rhs}

    exact = _parse_date_token(expr)
    if exact is not None:
        return {"mode": "exact", "rhs": exact}

    return {"mode": "substring", "expr": expr}


def _date_filter_matches(session_name, parsed_filter):
    mode = parsed_filter["mode"]
    if mode == "all":
        return True
    if mode == "substring":
        expr = parsed_filter["expr"]
        if fnmatch.fnmatch(session_name, f"*{expr}*"):
            return True
        session_date = _extract_session_date(session_name)
        return session_date is not None and expr in session_date.isoformat()

    session_date = _extract_session_date(session_name)
    if session_date is None:
        return False

    if mode == "exact":
        return session_date == parsed_filter["rhs"]
    if mode == "cmp":
        rhs = parsed_filter["rhs"]
        op = parsed_filter["op"]
        if op == ">=":
            return session_date >= rhs
        if op == "<=":
            return session_date <= rhs
        if op == ">":
            return session_date > rhs
        if op == "<":
            return session_date < rhs
        return False
    if mode == "range":
        lo = parsed_filter["lo"]
        hi = parsed_filter["hi"]
        if lo is not None and session_date < lo:
            return False
        if hi is not None and session_date > hi:
            return False
        return True
    return False


session_filter = os.environ.get("ADAMACS_SESSION_FILTER", "*")
date_filter = os.environ.get("ADAMACS_DATE_FILTER", "*")
parsed_date_filter = _parse_date_filter(date_filter)

root_dirs = dj.config.get("custom", {}).get("exp_root_data_dir", [])
if not root_dirs:
    raise ValueError('dj.config["custom"]["exp_root_data_dir"] is not configured.')

dataroot = root_dirs[0]
all_session_dirs = [
    d for d in os.listdir(dataroot) if os.path.isdir(os.path.join(dataroot, d))
]

session_matched_dirs = [d for d in all_session_dirs if fnmatch.fnmatch(d, session_filter)]

dirs_root = [
    d
    for d in session_matched_dirs
    if _date_filter_matches(d, parsed_date_filter)
]

sorted_dirs_root = natsorted(dirs_root, reverse=True)

print(f"Data root: {dataroot}")
print(f"Filter: session={session_filter!r}, date={date_filter!r} ({parsed_date_filter['mode']})")
print(f"Found {len(sorted_dirs_root)} candidate sessions.")

if parsed_date_filter["mode"] != "substring":
    unmatched_dates = [d for d in session_matched_dirs if _extract_session_date(d) is None]
    if unmatched_dates:
        print(
            f"Note: {len(unmatched_dates)} session folders had no parseable date and were skipped by date comparison."
        )

for path in sorted_dirs_root:
    display(HTML(f'<a href="{os.path.join(dataroot, path)}" target="_blank">{path}</a>'))


## 4) Launch ingest GUI (opt-in)

Set `ADAMACS_LAUNCH_GUI=1` before running this cell to open the interactive GUI.


In [ ]:
LAUNCH_GUI = os.environ.get('ADAMACS_LAUNCH_GUI', '0') == '1'

if not sorted_dirs_root:
    print('No matching sessions found. Adjust ADAMACS_SESSION_FILTER / ADAMACS_DATE_FILTER and rerun.')
elif not LAUNCH_GUI:
    print('GUI launch skipped. Set ADAMACS_LAUNCH_GUI=1 to open the interactive selector.')
    print('First 10 matching sessions:')
    for path in sorted_dirs_root[:10]:
        print('  -', path)
else:
    print('ADAMACS INGEST GUI v2')
    selected_data, get_dlc_models = ai.select_sessions(
        sorted_dirs_root,
        do_population=False,
        rspace_upload=False,
        ingest_opt='trigger',
    )


## Notes

- Use this notebook as the default ingest entrypoint.
- Keep ad-hoc debugging in separate notebooks/scripts.
